Plan

1. Model
2. Learning rate schedule - manually set lr
3. Optimizer

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /project/dl2025s/velikanm/robomimic/

In [ ]:
#!pip install timm torch-summary

# Data

Robomimic dataset https://robomimic.github.io/docs/datasets/overview.html

# Dataloader

In [ ]:
from glob import glob
from pathlib import Path
import os

import numpy as np
import cv2
from skimage.io import imread

import torch
import torch.utils.data as data  # dataset shuffling, sample batching, ...
from torch import nn  # neural network layers
from torchvision import transforms # augmentation

# pretrained models
from torchvision.models import resnet18, get_model
import timm
import torchsummary

import matplotlib.pyplot as plt
import pickle # Save/load patch locations.


def imginfo(img):
    print(type(img), img.dtype, img.shape, img.min(), img.max())

In [ ]:
np.random.randint(low=0, high=15)

## Robotics dataset

In [ ]:
from datasest import RobomimicLoader

ACTION_DIM = 7
DATASET_PATH = "datasets/demo.hdf5"  # Change this to your dataset path
if not Path(DATASET_PATH).exists():
    print("Dataset file doesn't exist")

dataset = RobomimicLoader(DATASET_PATH, history_length=3)
image, action = dataset[0]
imginfo(image)
imginfo(action)
print(len(dataset))

# Model

See table of pretrained pytorch classifiers.

In [ ]:
explore_models = False

if explore_models:
    #model = get_model("efficientnet_v2_s", weights="EfficientNet_V2_S_Weights.IMAGENET1K_V1")
    model = get_model("mobilenet_v3_small", weights="MobileNet_V3_Small_Weights.IMAGENET1K_V1")

    torchsummary.summary(model, (3, 224, 224), device="cpu")
    print()

    print("model layers:", len( list(model.children()) ))
    backbone, avgpool, head = model.children()

    print("backbone layers:", len( list(backbone.children()) ))
    backbone_layers = list(backbone.children())

    in_feat = 40
    num_classes = 17
    backbone_reduced = nn.Sequential(
        *backbone_layers[:6],
        nn.Sequential(avgpool, nn.Flatten()),
        nn.Linear(in_feat, num_classes))
    
    torchsummary.summary(backbone_reduced, (3, 224, 224), device="cpu")
    print()

In [ ]:
class Vit(nn.Module):
    def __init__(self, input_shape=None, num_classes=17):
        super().__init__()

        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True)
        for param in self.vit.parameters():
            param.requires_grad = False
        self.vit.head = nn.Sequential(
            nn.Linear(self.vit.head.in_features, 512),
            nn.Hardswish(),
            nn.Dropout(p=0.2),
            nn.Linear(512, num_classes)
        )

    def forward(self, images):
        # images: (b, c, h, w)
        batch_size, channels, h, w = images.shape
        
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
        ])
        images = transform(images) # (c, h, w)
        
        # if input image has 1 channel, repeat the torch tensor
        if channels < 3:
            images = images.repeat(1, 3, 1, 1)

        x = self.vit(images) # (b, c, h, w) -> (b, 768) -> (b, num_classes)
        return x

In [ ]:
from imitation_model import CNN

model = CNN(input_channels=9, n_classes=ACTION_DIM)

#model = Vit(num_classes=ACTION_DIM)

# torchsummary.summary(model, (9, 224, 224), device="cpu")
# print()

Check that the model works and the output has correct shape.

In [ ]:
image, action = dataset[0]
image = image[None, ...]
print("training sample")
imginfo(image)
print("model output:", model(image))
#assert model(image).shape == (1, NUM_CLASSES)

# Training

Manual learning rate schedule.

In [ ]:
# hint: tpus in colab
EPOCHS = 10


# rampup + sustain
rampup_epochs = 0
start_lr = 0.0001
sustain_epochs = 0

# exponential decay
min_lr = 0.001
max_lr = 0.01
exp_decay = .8

# step
step_epoch = []
step_lr = []

def lrfn(epoch):
  if epoch < rampup_epochs:
    return (max_lr - start_lr)/rampup_epochs * epoch + start_lr
  elif epoch < rampup_epochs + sustain_epochs:
    return max_lr
  else:
    for i in range(len(step_epoch)-1, -1, -1):
        if epoch > step_epoch[i]:
            return step_lr[i]
    return (max_lr - min_lr) * exp_decay**(epoch-rampup_epochs-sustain_epochs) + min_lr

t = np.arange(EPOCHS)
y = [lrfn(x) for x in t]
plt.plot(t, y)
print('Learning rate per epoch:')

In [ ]:
from tqdm import tqdm

DEVICE = "cuda:0"

def train(model, train_loader, val_loader, criterion, optimizer, num_epochs):
    train_losses = []
    val_losses = []

    for epoch in range(num_epochs):
        # Training
        model.train()
        running_loss = 0.0

        for g in optimizer.param_groups:
            g['lr'] = lrfn(epoch)

        for inputs, labels in tqdm(train_loader):
            # (b, c, h, w), (b)
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'Train Loss: {train_loss:.4f}')

        # Validation
        if False:
            model.eval()
            running_loss = 0.0

            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)

                    running_loss += loss.item()

            val_loss = running_loss / len(val_loader)
            
            print(f'Val Loss: {val_loss:.4f}')

        if False:
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), f'best_{model.__class__.__name__}.pth')


In [ ]:
def main():
  load_ckpt = False

  num_epochs = EPOCHS
  batch_size = 256
  workers = 2

  #dataset = FlowerDataset()
  dataset = RobomimicLoader(DATASET_PATH)

  #criterion = nn.CrossEntropyLoss()
  criterion = nn.MSELoss()
  model = Vit(num_classes=ACTION_DIM).to(DEVICE)
  if load_ckpt:
      ckpt = torch.load("fc.pth", map_location=torch.device(DEVICE))
      model.fc = ckpt["fc"]

  optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)


  train_loader = data.DataLoader(dataset, batch_size=batch_size, shuffle=True,
      num_workers=workers, drop_last=False, pin_memory=False)


  train(model, train_loader, train_loader, criterion, optimizer, num_epochs)


main()

In [ ]:
image, action = dataset[3]
image = image[None, ...]
print("training sample")
imginfo(image)

out = model(image)[0].detach().to('cpu')
print("model output:")
print(out)
print("gt:")
print(action)

thr = 0.2
print((out  - action).abs() < thr)
#assert model(image).shape == (1, NUM_CLASSES)

# STOP

In [ ]:
DEVICE = "cuda" # "cpu" or "cuda"
PRINT_FREQ = 50 # how frequently to print training status
VAL_FREQ = 250 # how frequently to run validation

from torch.utils.tensorboard import SummaryWriter

class Logger:
    def __init__(self, path='runs/logbook', total_steps=0):
        self.total_steps = total_steps
        self.running_loss = {}
        self.writer = None
        self.writer_path = path

    def _print_training_status(self):
        # write average values to tensorboard
        if self.writer is None:
            self.writer = SummaryWriter(self.writer_path)
        for key in self.running_loss:
            self.writer.add_scalar(key, self.running_loss[key]/PRINT_FREQ, self.total_steps)
            self.running_loss[key] = 0.0

    def push(self, metrics):
        # sum new metric values for averaging
        self.total_steps += 1

        for key in metrics:
            if key not in self.running_loss:
                self.running_loss[key] = 0.0
            self.running_loss[key] += metrics[key]

        if self.total_steps % PRINT_FREQ == PRINT_FREQ - 1:
            self._print_training_status()
            self.running_loss = {}

    def write_dict(self, results):
        # write values to tensorboard without averaging
        if self.writer is None:
            self.writer = SummaryWriter(self.writer_path)
        for key in results:
            self.writer.add_scalar(key, results[key], self.total_steps)

    def close(self):
        self.writer.close()

In [ ]:
import time
import datetime as dt

def calc_validation_score():
    model.train(False)
    errors = []
    for image_id in val_image_ids:
        errors.append(calc_error(image_id))
    model.train(True)
    return np.mean(np.array(errors))

def train(model, train_dl, optimizer, logger, visualize_list, train_steps):
    """
    Main training loop. Load training data, run model forward and backward pass,
    print training status, run validation.
    """
    start = time.time()
    model.to(DEVICE)

    softmax = nn.Softmax(dim=-1).to(DEVICE)
    eps = torch.tensor(0.0001).to(DEVICE)
    binary_cross_entropy = nn.BCELoss(reduction="sum")

    print("Training steps:", train_steps)

    while True:
        model.train(True)  # set training mode
        dataloader = train_dl
        step = 0
        ema_loss = 0.0
        start_time = time.time()

        # iterate over data
        for left_patch, right_strip, labels in dataloader:
            step += 1

            # moving tensors to GPU
            # tensor shape b h w c -> b c h w
            left_patch = left_patch.to(DEVICE).permute(0, 3, 1, 2).float().contiguous() # contiguous speeds up 2-4x
            right_strip = right_strip.to(DEVICE).permute(0, 3, 1, 2).float().contiguous()
            labels = labels.to(DEVICE).contiguous() # shape=(b w)

            # forward pass
            optimizer.zero_grad()

            # Run model on left_patch, right_strip. Compute loss with labels.

            # your code here
            # vvvvvvvvvvvvvv

            patch_feat = model(left_patch) # shape=(b c 1 1)
            strip_feat = model(right_strip) # shape=(b c 1 w)
            inner_prod = torch.sum(patch_feat * strip_feat, dim=1) # broadcasting. shape=(b 1 w)
            inner_prod = torch.squeeze(inner_prod) # remove all dimensions of size 1. shape=(b w)
            target = softmax(inner_prod) # softmax by last dimension
            target = target * (1 - eps * 2) + eps # do not allow too high loss
            target_view = target.view(-1).float() # BCE loss does not allow complex shape
            labels_view = labels.view(-1).float()
            loss = binary_cross_entropy(target_view, labels_view)
            # loss = -torch.sum(labels * torch.log(target + eps) + (1 - labels) * torch.log(1 - target + eps))

            # ^^^^^^^^^^^^^^

            # backward pass
            loss.backward()
            optimizer.step()

            show_loss = loss.detach().cpu().numpy()
            ema_loss = show_loss * (2/(1+step)) + ema_loss * (1-2/(1+step))
            metrics = {
                'cross_entropy_loss': show_loss
            }
            logger.push(metrics)

            if step % PRINT_FREQ == PRINT_FREQ - 1:
                print("step: {}, EMA BCE loss: {:.2f}, step time: {:.2f}".format(step,
                                                                                 ema_loss, (time.time() - start_time) / step))
                left_img = imread(left_image_paths[0]).astype(np.float64) / 255.0
                right_img = imread(right_image_paths[0]).astype(np.float64) / 255.0
                out_img = evaluate_model(model, left_img, right_img)
                visualize_list.append((out_img, step))

            if step % VAL_FREQ == VAL_FREQ - 1:
                val_error = calc_validation_score()
                metrics = {
                    'val_error_percent': val_error
                }
                logger.write_dict(metrics)
                print("validation error: {:.2f}".format(val_error))

            if step > train_steps:
                return

    return train_loss, valid_loss, disp_vis

experiment = dt.datetime.now().strftime("%H%M%S")
logger = Logger(path="runs/logbook-" + experiment)
visualize_list = [] # [(disparity image, steps performed), ...]
model = PatchModel()
print("Parameters:", sum(p.numel() for p in model.parameters()))
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
train_loader = data.DataLoader(train_dataset, batch_size=128,
        pin_memory=False, shuffle=True, num_workers=2, drop_last=True)

train(model, train_loader, optimizer, logger, visualize_list, train_steps=500)

# Augmentation

In [ ]:
import torchvision.transforms as T

torch.manual_seed(0)

def display_train_patches(idx):
    left_patch, right_strip_patch, labels = train_dataset[idx]
    print(left_patch.shape)

    for i in range(10):
        t = torch.clone(right_strip_patch.permute(2, 0, 1))

        # your code here
        # vvvvvvvvvvvvvv
        photo_aug = T.Compose([
                   T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.5/3.14),
                   T.GaussianBlur(7)
        ])
        t = photo_aug(t)
        # ^^^^^^^^^^^^^^

        img = t.permute(1, 2, 0)

        plt.figure(figsize=(15, 15))
        plt.imshow(img)
        plt.show()

display_train_patches(0)